# Notebook 04 — Loss Function Verification
**Run after:** `03_build_model.ipynb`

This notebook shows exactly what each loss function does and why.
No training, no GPU required — just understanding the math.

| Loss | Purpose |
|---|---|
| Focal Loss | Down-weights easy samples (nevus), focuses on rare/hard (melanoma) |
| Evidential Loss | Penalizes confident-wrong predictions — core medical safety property |
| Combined Loss | Both together — correct AND calibrated predictions |


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
import torch
import torch.nn.functional as F
import numpy as np
import math
import pandas as pd

from models.losses import FocalLoss, EvidentialLoss, CombinedLoss
from models.trainer import get_device

device = get_device()
NUM_CLASSES = 7


  Device: Apple M2 GPU (MPS) ✅ — training will use GPU cores


## Focal Loss — The Focusing Effect
Standard cross-entropy treats all examples equally.
Focal loss multiplies each loss by `(1 - p_true)^γ`:
- Easy example (p=0.95): `(1-0.95)^2 = 0.0025` → loss nearly zeroed out
- Hard example (p=0.20): `(1-0.20)^2 = 0.64` → loss barely reduced

This means **gradient signal comes overwhelmingly from rare, hard classes**.


In [2]:
focal_fn = FocalLoss(gamma=2.0)
rows = []
scenarios = [
    ("Easy — nevus, very confident", 0.95),
    ("Moderate — 70% confidence",    0.70),
    ("Uncertain — 50/50",            0.50),
    ("Hard — melanoma, 20% prob",    0.20),
    ("Very wrong — 5% true class",   0.05),
]
for name, p_true in scenarios:
    x      = math.log(max(p_true * (NUM_CLASSES-1) / (1-p_true+1e-9), 1e-9))
    logits = torch.tensor([[x, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
    target = torch.tensor([0])
    ce_val    = F.cross_entropy(logits, target).item()
    focal_val = focal_fn(logits, target).item()
    rows.append({'Scenario': name, 'p_true': p_true,
                 'CE Loss': round(ce_val,4), 'Focal Loss': round(focal_val,4),
                 'Reduction': f"{focal_val/max(ce_val,1e-9):.3f}×"})

df_focal = pd.DataFrame(rows)
df_focal.style.background_gradient(subset=['Focal Loss'], cmap='RdYlGn_r')


,Scenario,p_true,CE Loss,Focal Loss,Reduction
0,"Easy — nevus, very confident",0.950000,0.051300,0.000100,0.003×
1,Moderate — 70% confidence,0.700000,0.356700,0.032100,0.090×
2,Uncertain — 50/50,0.500000,0.693100,0.173300,0.250×
3,"Hard — melanoma, 20% prob",0.200000,1.609400,1.030000,0.640×
4,Very wrong — 5% true class,0.050000,2.995700,2.703600,0.902×


## Evidential Loss — Penalizing Confident Wrong Predictions
The model outputs Dirichlet alpha parameters.
Think of alpha as an "evidence counter" for each class.

- `[10, 1, 1, 1, 1, 1, 1]` = lots of evidence for class 0 → confident
- `[2, 2, 2, 2, 2, 2, 2]` = uniform evidence → uncertain
- `[1, 10, 1, 1, 1, 1, 1]` = confident about the **wrong** class → penalized heavily


In [3]:
ev_fn  = EvidentialLoss(num_classes=NUM_CLASSES)
target = torch.tensor([0])   # true class is 0

rows = []
scenarios = [
    ("High evidence correct [10,1,1,1,1,1,1]", torch.tensor([[10.,1.,1.,1.,1.,1.,1.]])),
    ("Moderate correct [5,1,1,1,1,1,1]",        torch.tensor([[5.,1.,1.,1.,1.,1.,1.]])),
    ("Uncertain [2,2,2,2,2,2,2]",               torch.tensor([[2.,2.,2.,2.,2.,2.,2.]])),
    ("Wrong class confident [1,10,1,1,1,1,1]",  torch.tensor([[1.,10.,1.,1.,1.,1.,1.]])),
    ("Very confident wrong [1,100,1,1,1,1,1]",  torch.tensor([[1.,100.,1.,1.,1.,1.,1.]])),
]
for name, alpha in scenarios:
    loss = ev_fn(alpha, target, epoch=10, max_epochs=20).item()
    rows.append({'Scenario': name, 'Evidential Loss': round(loss, 4)})

df_ev = pd.DataFrame(rows)
df_ev.style.background_gradient(subset=['Evidential Loss'], cmap='RdYlGn_r')


,Scenario,Evidential Loss
0,"High evidence correct [10,1,1,1,1,1,1]",0.470000
1,"Moderate correct [5,1,1,1,1,1,1]",0.788500
2,"Uncertain [2,2,2,2,2,2,2]",2.734600
3,"Wrong class confident [1,10,1,1,1,1,1]",6.887400
4,"Very confident wrong [1,100,1,1,1,1,1]",20.065800


## The Core Medical Safety Property
Two models, both wrong about melanoma (true class = 1):
- **Model A**: 95% sure it's nevus → DANGEROUS
- **Model B**: 50/50 between nevus and melanoma → ACCEPTABLE

Our combined loss penalizes Model A much more than Model B.


In [5]:
true_class = torch.tensor([1]).to(device)  # true = melanoma

class_counts = {0:4800, 1:800, 2:800, 3:370, 4:235, 5:100, 6:82}
combined_fn  = CombinedLoss(num_classes=7, gamma=2.0, lambda_ev=0.5,
                             class_counts=class_counts, device=device)

# Model A: 95% confident it's nevus (WRONG)
logits_A = torch.tensor([[4.0, 0.2, 0., 0., 0., 0., 0.]]).to(device)
alpha_A  = torch.tensor([[10., 1., 1., 1., 1., 1., 1.]]).to(device)

# Model B: uncertain between nevus and melanoma
logits_B = torch.tensor([[1.5, 1.3, 0., 0., 0., 0., 0.]]).to(device)
alpha_B  = torch.tensor([[3., 3., 1., 1., 1., 1., 1.]]).to(device)

loss_A = combined_fn({'logits': logits_A, 'alpha': alpha_A}, true_class, epoch=15, max_epochs=30)
loss_B = combined_fn({'logits': logits_B, 'alpha': alpha_B}, true_class, epoch=15, max_epochs=30)

pA = F.softmax(logits_A, dim=1)[0]
pB = F.softmax(logits_B, dim=1)[0]

print(f"{'':30} {'Model A (confident wrong)':>25} {'Model B (uncertain wrong)':>26}")
print(f"{'─'*30} {'─'*25} {'─'*26}")
print(f"{'P(nevus)':30} {pA[0].item():>25.3f} {pB[0].item():>26.3f}")
print(f"{'P(melanoma)':30} {pA[1].item():>25.3f} {pB[1].item():>26.3f}")
print(f"{'Focal loss':30} {loss_A['focal']:>25.4f} {loss_B['focal']:>26.4f}")
print(f"{'Evidential loss':30} {loss_A['evidential']:>25.4f} {loss_B['evidential']:>26.4f}")
print(f"{'TOTAL loss':30} {loss_A['total'].item():>25.4f} {loss_B['total'].item():>26.4f}")
print()
print("✅ Model A gets a HIGHER total loss — the system penalizes confident-wrong more.")
print("   This is the medical safety property: uncertainty is better than false confidence.")
print("\n→ NEXT: Open notebook 05_train.ipynb")


  Focal alpha (per-class weights): [0.007 0.039 0.039 0.085 0.134 0.314 0.383]
  Combined Loss: Focal(γ=2.0) + 0.5×Evidential
                               Model A (confident wrong)  Model B (uncertain wrong)
────────────────────────────── ───────────────────────── ──────────────────────────
P(nevus)                                           0.898                      0.341
P(melanoma)                                        0.020                      0.279
Focal loss                                        0.1472                     0.0260
Evidential loss                                   6.8874                     2.1958
TOTAL loss                                        3.5909                     1.1239

✅ Model A gets a HIGHER total loss — the system penalizes confident-wrong more.
   This is the medical safety property: uncertainty is better than false confidence.

→ NEXT: Open notebook 05_train.ipynb
